# 03 -- Starting Pitcher Stats Ingestion

Fetch starting pitcher statistics from FanGraphs for seasons 2015-2024.

**Requirement:** DATA-03

**Source:** FanGraphs via `pybaseball.pitching_stats()`

**Key columns:** Name, Team, W, L, ERA, GS, IP, FIP, xFIP, SIERA, K%, BB%, WHIP, WAR

**Filter:** Only pitchers with GS >= 1 (at least one start). Cache key includes min_gs threshold.

In [ ]:
# If editable install is not set up, uncomment the next line:
# import sys; sys.path.insert(0, "..")

from src.data.sp_stats import fetch_sp_stats
from src.data.cache import load_manifest

In [ ]:
# Fetch starting pitcher stats for all 10 backtest seasons
seasons = range(2015, 2025)
dfs = {}
for season in seasons:
    print(f"Fetching {season}...")
    dfs[season] = fetch_sp_stats(season)
    print(f"  {len(dfs[season])} starting pitchers (GS >= 1)")

In [ ]:
# Summary: show key pitching metrics from 2023
df_sample = dfs[2023]
print(f"Columns ({len(df_sample.columns)} total): {list(df_sample.columns)[:20]}...")
print(f"\nKey starting pitcher metrics (2023, top 10 by IP):")
top_sp = df_sample.nlargest(10, "IP")
display(top_sp[["Name", "Team", "GS", "IP", "ERA", "FIP", "xFIP", "K%", "BB%", "WHIP"]].reset_index(drop=True))

print(f"\nFIP range: {df_sample['FIP'].min():.2f} - {df_sample['FIP'].max():.2f}")
print(f"xFIP range: {df_sample['xFIP'].min():.2f} - {df_sample['xFIP'].max():.2f}")

In [ ]:
# Coverage validation: reasonable pitcher counts per season
# Expect 100-200 starting pitchers per season (30 teams x 3-7 SPs)
for season, df in dfs.items():
    n_pitchers = len(df)
    min_gs = df["GS"].min()
    max_gs = df["GS"].max()
    flag = "OK" if 80 <= n_pitchers <= 300 else "CHECK"
    short_flag = " (shortened)" if df["is_shortened_season"].iloc[0] else ""
    print(f"{season}: {n_pitchers} SPs, GS range {min_gs}-{max_gs} [{flag}]{short_flag}")

In [ ]:
# Cache check
manifest = load_manifest()
sp_keys = {k: v for k, v in manifest.items() if k.startswith("sp_stats_")}
print(f"Cached SP stats files: {len(sp_keys)}")
for k, v in sorted(sp_keys.items()):
    print(f"  {k}: {v['row_count']} rows, fetched {v['fetch_date'][:10]}")